# Tags — Topic Tag Generation

Runs tag generation across the vault using the configured backend and compares
results against the ground truth in `data/ground_truth.yaml`.

In [1]:
import sys
sys.path.insert(0, '../src')

from glob import glob
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from notemine.config import load_config
from notemine.frontmatter_io import load_note
from notemine.ground_truth import load_ground_truth
from notemine.tasks import run_tags

load_dotenv('../.env')
config = load_config('../config.toml')
gt = load_ground_truth('../' + config['vault']['ground_truth'].lstrip('./'))

vault = '../' + config['vault']['path'].lstrip('./')
config['vault']['prompts_dir'] = '../' + config['vault']['prompts_dir'].lstrip('./')

notes = sorted(
    p for p in glob(f'{vault}/**/*.md', recursive=True)
    if not Path(p).name.startswith('index_')
)
print(f'{len(notes)} notes found')

20 notes found


## Run tag generation

In [ ]:
backend = config['run']['backend']

results = []
for path in notes:
    name = Path(path).name
    post = load_note(path)
    if len(post.content.strip()) < 50:
        continue

    gt_entry = gt.get(name, {})
    gt_tags = set(gt_entry.get('tags', []))

    row = {'file': name, 'lang': gt_entry.get('lang', '?'), 'ground_truth': sorted(gt_tags)}

    try:
        row[backend] = run_tags(backend, post.content, config)
    except Exception as e:
        row[backend] = f'ERROR: {e}'

    results.append(row)
    print(f'  {name}')

print(f'\nDone — {len(results)} notes processed')

## Summary comparison

In [ ]:
def tag_stats(gt_tags, pred):
    if isinstance(pred, str):
        return pred
    gt_set = set(gt_tags)
    pred_set = set(pred)
    matched = len(gt_set & pred_set)
    extra = len(pred_set - gt_set)
    pct = int(100 * matched / len(gt_set)) if gt_set else 0
    return f'{matched}/{len(gt_set)} ({pct}%)  +{extra} extra'

rows = []
for r in results:
    rows.append({
        'file': r['file'],
        'lang': r['lang'],
        'ground_truth': r['ground_truth'],
        backend: tag_stats(r['ground_truth'], r[backend]),
    })

pd.set_option('display.max_colwidth', 40)
pd.DataFrame(rows)

## Inspect a single note

Change `INSPECT` to any filename to see the full tag lists side by side.

In [ ]:
INSPECT = Path(notes[0]).name

row = next((r for r in results if r['file'] == INSPECT), None)
if row is None:
    print(f'{INSPECT} not found')
else:
    gt_set = set(row['ground_truth'])
    pred = set(row[backend]) if not isinstance(row[backend], str) else set()

    all_tags = sorted(gt_set | pred)
    tag_rows = [{
        'tag': t,
        'ground_truth': '✓' if t in gt_set else '',
        backend: '✓' if t in pred else '',
    } for t in all_tags]

    print(f'=== {INSPECT} ===')
    display(pd.DataFrame(tag_rows))

## (Optional) Save results to frontmatter

Uncomment to write results under `notemine.tags.<backend>`.

In [ ]:
# from notemine.frontmatter_io import save_note
#
# for path in notes:
#     name = Path(path).name
#     row = next((r for r in results if r['file'] == name), None)
#     if row is None:
#         continue
#     post = load_note(path)
#     notemine = post.metadata.get('notemine', {})
#     tags_data = notemine.get('tags', {})
#     if not isinstance(row.get(backend), str):
#         tags_data[backend] = row[backend]
#     notemine['tags'] = tags_data
#     post.metadata['notemine'] = notemine
#     save_note(path, post)
# print('Saved.')